# Xbar-S Analysis

Xbar-S charts are the gold standard for monitoring processes with subgrouped data. They're ideal when you have:

- Multiple measurements per time period
- Rational subgroups (factors like machines, operators, batches)
- Need to monitor both location (mean) and spread (variation)

## What You'll Learn

1. Create Xbar and S charts from replicated data
2. Understand how SDS affects variance estimation
3. Compare factor levels using control charts
4. Access VAS residuals for deeper analysis

## Setup

In [ ]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame

## Create Replicated Data

We'll simulate a filling machine with:
- 3 operators (A, B, C)
- 8 time periods
- 4 replicate measurements per operator per time period

This creates **SDS 1: Full Replication** - the most powerful design.

In [ ]:
np.random.seed(42)

operators = ['A', 'B', 'C']
n_times = 8
n_reps = 4

data = []
for t in range(n_times):
    for op in operators:
        # Each operator has a slightly different mean
        op_effect = {'A': 0, 'B': 2, 'C': -1}[op]
        
        # Add time trend (process drift)
        time_effect = t * 0.3
        
        for rep in range(n_reps):
            # Add special cause for Operator B at time 6
            special = 8 if (op == 'B' and t == 6) else 0
            
            value = 100 + op_effect + time_effect + special + np.random.normal(0, 1.5)
            data.append({
                'time': t + 1,
                'operator': op,
                'weight': round(value, 2)
            })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} observations")
print(f"Structure: {len(operators)} operators x {n_times} times x {n_reps} reps")
df.head(12)

## Formulate the Study

In [ ]:
pdf = ProcessDataFrame(df)

study = pdf.formulate(
    response=pdf.columns.weight,
    factors=[pdf.columns.operator],
    time=pdf.columns.time
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Description: {study.sds_description}")
print(f"\nValid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")
print(f"Residual charts: {study.residual_charts}")

## Understanding SDS 1

**SDS 1 (Full Replication)** is the most powerful sampling design because:

1. Every (operator, time) cell has multiple observations
2. Within-cell variance can be estimated exactly
3. All VAS residuals (R1-R5) are available
4. Interactions can be detected

The formula for control limits uses the pooled within-cell standard deviation.

## Analyze with Xbar-S Charts

In [ ]:
result = study.analyze()  # Uses recommended chart (Xbar)

print(f"Charts created: {result.all_charts}")
print(f"Has residuals: {result.has_residuals}")

## View Chart Data

In [ ]:
# Xbar chart shows subgroup means
xbar_data = result.get_chart('Xbar')
print("Xbar Chart Data (subgroup means):")
xbar_data.head(10)

In [ ]:
# S chart shows subgroup standard deviations
s_data = result.get_chart('S')
print("\nS Chart Data (subgroup std devs):")
s_data.head(10)

In [ ]:
# Statistics for both charts
print("Xbar Statistics:")
display(result.get_statistics('Xbar'))

print("\nS Statistics:")
display(result.get_statistics('S'))

## Visualize Xbar Chart

In [ ]:
fig = result.plot(
    chart='Xbar',
    show_zones=True,
    show_signals=True,
    show_stats=True
)
fig.show()

## Visualize S Chart

In [ ]:
fig = result.plot(
    chart='S',
    show_zones=True,
    show_signals=True
)
fig.show()

## Understanding Xbar-S Charts

### The Xbar Chart

- Plots the **mean** of each subgroup
- Centerline: Grand mean of all observations
- Limits based on within-subgroup variation
- Detects **shifts in process level**

### The S Chart

- Plots the **standard deviation** of each subgroup
- Centerline: Pooled within-subgroup standard deviation
- Limits based on chi-square distribution
- Detects **changes in process variation**

### Reading Order

1. **First check the S chart** - Variation must be stable
2. **Then interpret the Xbar chart** - Only valid if S is stable
3. Points on Xbar beyond limits → investigate the specific subgroup

## Signal Detection for Xbar-S

For Xbar and S charts (categorical comparisons), only **Rule 1** applies - points beyond the control limits.

In [ ]:
# Detect signals on Xbar
signals = result.detect_signals(chart='Xbar')

print(f"Xbar signals: {signals.count}")
if signals.has_signals:
    print("\nViolations:")
    display(signals.violations)

In [ ]:
# Detect signals on S
signals_s = result.detect_signals(chart='S')

print(f"S chart signals: {signals_s.count}")

## Accessing VAS Residuals

With SDS 1 (full replication), all five residuals are available:

In [ ]:
# View the computed residuals
residuals = result.residuals
print("VAS Residuals:")
residuals.head(10)

In [ ]:
# Analyze time effects using R4
result_r4 = study.analyze(chart='R4_Imr')

fig = result_r4.plot(
    show_zones=True,
    show_signals=True,
    title='R4 Residuals: Time Effects'
)
fig.show()

In [ ]:
# Analyze factor (operator) effects using R5
result_r5 = study.analyze(chart='R5_Imr')

fig = result_r5.plot(
    show_zones=True,
    show_signals=True,
    title='R5 Residuals: Operator Effects'
)
fig.show()

## Chart Table Summary

Get a compact summary table for reporting:

In [ ]:
# Summary table with subgroup info, values, and limits
table = result.chart_table('Xbar')
table

## Summary

In this tutorial, you learned:

- Xbar-S charts require subgrouped data (n >= 2 per cell)
- SDS 1 (full replication) provides the most analytical power
- The S chart monitors variation; the Xbar chart monitors level
- Only Rule 1 applies to Xbar-S charts
- VAS residuals enable deeper root cause analysis

## Next Steps

- [Stratified Analysis](stratified-analysis.ipynb) - Separate charts per factor level
- [VAS Residuals](../user-guide/residuals.md) - Deep dive into VAS residuals
- [Signal Detection](signal-detection.ipynb) - All Western Electric rules